# SCREEN: metadata для homologous ELS cCRE

Из корня проекта установите зависимости: `python -m pip install -r requirements.txt`.
Входы — homologous pairs без заголовка и human/mouse ELS BED из одной версии Registry.
В BED используется пятая колонка (`EH38E…` / `EM10E…`), а не rDHS из четвёртой.
Сохраняются связи one-to-many, удаляются только точные дубликаты пар.
API key можно получить в [SCREEN API Console](https://console.wenglab.org/docs/getting-started/auth).


In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / "eclare").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from eclare.data.screen import (
    load_orthologous_els, fetch_ccres, parse_zscores, fetch_human_mouse_metadata,
)

homologs_path = project_root / "data/encode/hg38-mm10-Homologous.tsv"
human_els_path = project_root / "data/encode/GRCh38-cCREs.ELS.bed"
mouse_els_path = project_root / "data/encode/mm10-cCREs.ELS.bed"


In [ ]:
import os
from getpass import getpass

# Ввод скрыт; ключ остаётся только в окружении текущего kernel.
if not os.environ.get("SCREEN_API_KEY"):
    os.environ["SCREEN_API_KEY"] = getpass("SCREEN API key: ")


Сначала проверим API на трёх уникальных cCRE каждого вида. Эта ячейка не запускает полную загрузку.


In [ ]:
preview_pairs = load_orthologous_els(homologs_path, human_els_path, mouse_els_path)
print(f"ELS pairs: {len(preview_pairs):,}")
print(preview_pairs.nunique())

for column, assembly in [("human_id", "grch38"), ("mouse_id", "mm10")]:
    ids = sorted(preview_pairs[column].unique())[:3]
    preview = parse_zscores(fetch_ccres(ids, assembly, batch_size=200))
    print(assembly, preview.shape)
    display(preview.head())


После просмотра ответа запустите полную загрузку. Главная функция также проверяет первые три ID обоих видов перед остальными запросами.
По умолчанию: `timeout=60`, `retries=3`, `backoff=1.0`, `cache_dir=".screen_cache"`.
Успешные батчи атомарно сохраняются в JSON; после сбоя повторите вызов с теми же файлами и `batch_size`.
Кеш учитывает assembly, endpoint, запрос и IDs; очистка кеша вручную или `cache_dir=None` позволяют обновить данные.
Нет фильтра по assay, biosample или Z-score. Одна строка — один assay/experiment/file для cCRE; пустой `zscores` не даёт строк.


In [ ]:
pairs, human_meta, mouse_meta = fetch_human_mouse_metadata(
    homologs_path,
    human_els_path,
    mouse_els_path,
    batch_size=200,
)

display(pairs.head())
display(human_meta.head())
display(mouse_meta.head())


In [ ]:
pairs.to_parquet("orthologous_els.parquet", index=False)
human_meta.to_parquet("human_ccre_metadata.parquet", index=False)
mouse_meta.to_parquet("mouse_ccre_metadata.parquet", index=False)
